# PCM Results Comparison Across Commitment Period Configurations

Compare Prescient PCM (production cost model) results for GTEP solutions with
different commitment period lengths: **1HR**, **2HR**, and **4HR**.

**Sections:**
1. Setup & Data Loading
2. GTEP Investment Decisions
3. Overall PCM Metrics Dashboard
4. Generation Mix by Fuel Type
5. LMP Analysis
6. Supply–Demand Balance
7. Operational Violations (load shedding, over-generation, reserve shortfall, line violations)
8. Renewables Curtailment
9. Computational Performance
10. Summary

In [ ]:
import json
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 5)})

# ── Paths ─────────────────────────────────────────────────────────────────
REPO_ROOT = Path.cwd().parent.parent  # from gtep/pcm_analysis/ -> repo root

CONFIGS = {
    '1HR': REPO_ROOT / 'retirement_allowed_no_extreme_full_load',
    '2HR': REPO_ROOT / 'retirement_allowed_no_extreme_2hr_commit',
    '4HR': REPO_ROOT / 'retirement_allowed_no_extreme_4hr_commit',
}

STAGE = '2035'  # which investment stage's PCM to analyze
PCM_SUBDIR = f'Prescient_2_{STAGE}'

FUEL_COLORS = {
    'NUC': '#e41a1c', 'COAL': '#555555', 'CT': '#ff7f00',
    'HYDRO': '#377eb8', 'WIND': '#4daf4a', 'PV': '#ffff33',
}
CONFIG_COLORS = {'1HR': '#1f77b4', '2HR': '#ff7f0e', '4HR': '#2ca02c'}

for label, p in CONFIGS.items():
    results = p / PCM_SUBDIR / 'results'
    print(f'{label}: {results}  exists={results.exists()}')

In [ ]:
def prescient_to_df(file_path):
    """Load Prescient CSV, combining Date/Hour/Minute into a Datetime column."""
    df = pd.read_csv(file_path)
    if 'Minute' in df.columns:
        df['Datetime'] = (
            pd.to_datetime(df['Date'])
            + pd.to_timedelta(df['Hour'], 'hour')
            + pd.to_timedelta(df['Minute'], 'minute')
        )
        df.drop(columns=['Date', 'Hour', 'Minute'], inplace=True)
    elif 'Hour' in df.columns:
        df['Datetime'] = (
            pd.to_datetime(df['Date'])
            + pd.to_timedelta(df['Hour'], 'hour')
        )
        df.drop(columns=['Date', 'Hour'], inplace=True)
    else:
        df['Datetime'] = pd.to_datetime(df['Date'])
        df.drop(columns=['Date'], inplace=True)
    cols = ['Datetime'] + [c for c in df.columns if c != 'Datetime']
    return df[cols]


def load_pcm_data(config_dir):
    """Load all PCM result CSVs for one configuration."""
    rd = config_dir / PCM_SUBDIR / 'results'
    data = {
        'overall': pd.read_csv(rd / 'overall_simulation_output.csv'),
        'daily': pd.read_csv(rd / 'daily_summary.csv', parse_dates=['Date']),
        'hourly_summary': prescient_to_df(rd / 'hourly_summary.csv'),
        'hourly_gen': prescient_to_df(rd / 'hourly_gen_summary.csv'),
        'bus': prescient_to_df(rd / 'bus_detail.csv'),
        'thermal': prescient_to_df(rd / 'thermal_detail.csv'),
        'renewables': prescient_to_df(rd / 'renewables_detail.csv'),
        'line': prescient_to_df(rd / 'line_detail.csv'),
        'runtimes': prescient_to_df(rd / 'runtimes.csv'),
        'gen': pd.read_csv(config_dir / PCM_SUBDIR / 'gen.csv'),
    }
    reserves_path = rd / 'reserves_detail.csv'
    if reserves_path.exists():
        data['reserves'] = prescient_to_df(reserves_path)
    return data


def load_gtep_results(config_dir):
    """Load GTEP investment JSON results."""
    gtep = {}
    for name in ['dispatchable_investments', 'renewable_investments', 'load_shed', 'costs']:
        path = config_dir / f'{name}.json'
        if path.exists():
            with open(path) as f:
                gtep[name] = json.load(f)
    return gtep


# Load everything
pcm = {}
gtep = {}
for label, cfg_dir in CONFIGS.items():
    print(f'Loading {label}...')
    pcm[label] = load_pcm_data(cfg_dir)
    gtep[label] = load_gtep_results(cfg_dir)
    n_days = len(pcm[label]['daily'])
    n_hours = len(pcm[label]['hourly_gen'])
    print(f'  {n_days} days, {n_hours} hours, '
          f'{len(pcm[label]["gen"])} generators')

N_DAYS = len(pcm['1HR']['daily'])
N_HOURS = len(pcm['1HR']['hourly_gen'])
print(f'\nSimulation period: {N_DAYS} days, {N_HOURS} hours')

## 2. GTEP Investment Decisions

Compare generator retirement/investment decisions across commitment period configs.

In [ ]:
print('GTEP Dispatchable Investment Decisions')
print('=' * 70)

for label in CONFIGS:
    inv = gtep[label].get('dispatchable_investments', {})
    stage_counts = {}
    for key, val in inv.items():
        stage = key.split('.')[0]
        if stage not in stage_counts:
            stage_counts[stage] = Counter()
        stage_counts[stage][val] += 1
    print(f'\n{label}:')
    for stage in sorted(stage_counts):
        c = stage_counts[stage]
        print(f'  {stage}: operational={c.get(True,0)}, retired={c.get(False,0)}')

print(f'\n\nGTEP Renewable Investment Decisions')
print('=' * 70)
for label in CONFIGS:
    inv = gtep[label].get('renewable_investments', {})
    stage_counts = {}
    for key, val in inv.items():
        stage = key.split('.')[0]
        if stage not in stage_counts:
            stage_counts[stage] = Counter()
        stage_counts[stage][val] += 1
    print(f'\n{label}:')
    for stage in sorted(stage_counts):
        c = stage_counts[stage]
        print(f'  {stage}: invested={c.get(True,0)}, not_invested={c.get(False,0)}')

# Check if decisions differ
inv_1hr = gtep['1HR'].get('dispatchable_investments', {})
inv_2hr = gtep['2HR'].get('dispatchable_investments', {})
inv_4hr = gtep['4HR'].get('dispatchable_investments', {})
if inv_1hr == inv_2hr == inv_4hr:
    print('\n** All three configs have IDENTICAL investment decisions. **')
    print('   PCM results will be driven by the same generator fleet.')
else:
    diffs = []
    for key in set(inv_1hr) | set(inv_2hr) | set(inv_4hr):
        vals = (inv_1hr.get(key), inv_2hr.get(key), inv_4hr.get(key))
        if len(set(vals)) > 1:
            diffs.append((key, vals))
    print(f'\n** {len(diffs)} generator(s) differ across configs: **')
    for key, vals in diffs[:20]:
        print(f'  {key}: 1HR={vals[0]}, 2HR={vals[1]}, 4HR={vals[2]}')

In [ ]:
# GTEP Load Shedding comparison
print('GTEP Load Shedding (from optimization)')
print('=' * 70)

for label in CONFIGS:
    ls = gtep[label].get('load_shed', {})
    total = sum(v for v in ls.values() if isinstance(v, (int, float)))
    nonzero = sum(1 for v in ls.values() if isinstance(v, (int, float)) and v > 0)
    print(f'{label}: total={total:>14,.2f} MWh, nonzero entries={nonzero}/{len(ls)}')

    # Parse by stage
    stage_shed = {}
    for key, val in ls.items():
        if not isinstance(val, (int, float)):
            continue
        stage = key.split('.')[0]
        stage_shed[stage] = stage_shed.get(stage, 0) + val
    for stage in sorted(stage_shed):
        print(f'  {stage}: {stage_shed[stage]:>14,.2f} MWh')

## 3. Overall PCM Metrics Dashboard

In [ ]:
rows = []
for label in CONFIGS:
    o = pcm[label]['overall'].iloc[0]
    bus = pcm[label]['bus']
    total_demand = bus['Demand'].sum()
    lw_lmp = (bus['Demand'] * bus['LMP DA']).sum() / total_demand if total_demand > 0 else 0

    rows.append({
        'Config': label,
        'Demand (TWh)': o['Total demand'] / 1e6,
        'Gen Costs ($M)': o['Total generation costs'] / 1e6,
        'Fixed Costs ($M)': o['Total fixed costs'] / 1e6,
        'Total Costs ($M)': o['Total costs'] / 1e6,
        'LW-LMP ($/MWh)': lw_lmp,
        'Avg Price ($/MWh)': o['Cumulative average price'],
        'Renew Pen (%)': o['Overall renewables penetration rate'],
        'Load Shed (MWh)': o['Total load shedding'],
        'Over-Gen (MWh)': o['Total over generation'],
        'Curtailment (MWh)': o['Total renewables curtailment'],
        'Reserve Shortfall (MWh)': o['Total reserve shortfall'],
        'Total On/Offs': o['Total on/offs'],
    })

dashboard = pd.DataFrame(rows).set_index('Config').T
print(f'PCM Metrics Dashboard — {N_DAYS} days')
print('=' * 75)
for metric in dashboard.index:
    vals = dashboard.loc[metric]
    line = f'  {metric:<28s}'
    for v in vals:
        if isinstance(v, float):
            line += f'{v:>14,.2f}'
        else:
            line += f'{v:>14,}'
    print(line)

# Check if all configs are identical
if dashboard['1HR'].equals(dashboard['2HR']) and dashboard['2HR'].equals(dashboard['4HR']):
    print('\n** All PCM results are IDENTICAL (same GTEP fleet -> same dispatch). **')

## 4. Generation Mix by Fuel Type

In [ ]:
for label in CONFIGS:
    gen = pcm[label]['gen']
    therm = pcm[label]['thermal']
    rnw = pcm[label]['renewables']

    gen_info = gen[['GEN UID', 'Unit Type', 'PMax MW']].copy()
    gen_info['GEN UID'] = gen_info['GEN UID'].astype(str)

    therm_m = therm.copy()
    therm_m['Generator'] = therm_m['Generator'].astype(str)
    therm_m = therm_m.merge(gen_info, left_on='Generator', right_on='GEN UID', how='left')
    thermal_gwh = therm_m.groupby('Unit Type')['Dispatch'].sum() / 1e3

    rnw_m = rnw.copy()
    rnw_m['Generator'] = rnw_m['Generator'].astype(str)
    rnw_m = rnw_m.merge(gen_info, left_on='Generator', right_on='GEN UID', how='left')
    renew_gwh = rnw_m.groupby('Unit Type')['Output'].sum() / 1e3

    all_gwh = pd.concat([thermal_gwh, renew_gwh]).sort_values(ascending=False)
    cap = gen.groupby('Unit Type')['PMax MW'].sum()
    cf = (all_gwh * 1e3) / (cap * N_HOURS) * 100

    print(f'\n=== {label} Generation Mix ({N_DAYS} days) ===')
    print(f'{"Type":<8s} {"Gen (GWh)":>12s} {"Share (%)":>10s} {"Cap (MW)":>10s} {"CF (%)":>8s}')
    print('-' * 52)
    for ut in all_gwh.index:
        gwh = all_gwh[ut]
        print(f'{ut:<8s} {gwh:>12,.1f} {gwh/all_gwh.sum()*100:>9.1f}% {cap.get(ut,0):>10,.0f} {cf.get(ut,0):>7.1f}%')
    print(f'{"TOTAL":<8s} {all_gwh.sum():>12,.1f}')

In [ ]:
# Daily generation mix stacked area — one subplot per config
fig, axes = plt.subplots(len(CONFIGS), 1, figsize=(14, 4 * len(CONFIGS)), sharex=True)
if len(CONFIGS) == 1:
    axes = [axes]

for ax, label in zip(axes, CONFIGS):
    gen = pcm[label]['gen']
    therm = pcm[label]['thermal'].copy()
    rnw = pcm[label]['renewables'].copy()
    gen_info = gen[['GEN UID', 'Unit Type']].copy()
    gen_info['GEN UID'] = gen_info['GEN UID'].astype(str)

    therm['Generator'] = therm['Generator'].astype(str)
    therm = therm.merge(gen_info, left_on='Generator', right_on='GEN UID', how='left')
    therm['date'] = therm['Datetime'].dt.normalize()
    daily_therm = therm.groupby(['date', 'Unit Type'])['Dispatch'].sum().unstack(fill_value=0) / 1e3

    rnw['Generator'] = rnw['Generator'].astype(str)
    rnw = rnw.merge(gen_info, left_on='Generator', right_on='GEN UID', how='left')
    rnw['date'] = rnw['Datetime'].dt.normalize()
    daily_rnw = rnw.groupby(['date', 'Unit Type'])['Output'].sum().unstack(fill_value=0) / 1e3

    daily_mix = pd.concat([daily_therm, daily_rnw], axis=1).fillna(0)
    order = [c for c in ['NUC', 'COAL', 'CT', 'HYDRO', 'WIND', 'PV'] if c in daily_mix.columns]
    daily_mix = daily_mix[order]
    colors = [FUEL_COLORS.get(c, '#999') for c in order]

    ax.stackplot(daily_mix.index, *[daily_mix[c] for c in order],
                 labels=order, colors=colors, alpha=0.85)
    ax.set_ylabel('GWh/day')
    ax.set_title(f'{label} — Daily Generation Mix')
    ax.legend(loc='upper right', ncol=len(order), fontsize=8)

axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.show()

## 5. LMP Analysis

In [ ]:
# Compute system load-weighted LMP for each config
sys_lmp = {}
for label in CONFIGS:
    bus = pcm[label]['bus']
    hourly_lw = bus.groupby('Datetime').apply(
        lambda g: (g['Demand'] * g['LMP DA']).sum() / g['Demand'].sum()
        if g['Demand'].sum() > 1.0 else g['LMP DA'].mean()
    ).rename('LW_LMP')
    sys_lmp[label] = hourly_lw

# Summary table
print(f'Load-Weighted LMP Summary ($/MWh) — {N_DAYS} days')
print('=' * 65)
print(f'{"Statistic":<20s}', end='')
for label in CONFIGS:
    print(f'{label:>14s}', end='')
print()
print('-' * 65)
for stat_name, stat_fn in [('Mean', 'mean'), ('Median', 'median'),
                            ('Std', 'std'), ('Min', 'min'), ('Max', 'max')]:
    print(f'{stat_name:<20s}', end='')
    for label in CONFIGS:
        val = getattr(sys_lmp[label], stat_fn)()
        print(f'${val:>12.2f}', end='')
    print()

for label in CONFIGS:
    neg = (sys_lmp[label] < 0).sum()
    print(f'Negative LMP hours ({label}): {neg} ({neg/len(sys_lmp[label])*100:.2f}%)')

In [ ]:
# LMP plots — overlay all configs
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (a) Hourly LMP timeseries
ax = axes[0, 0]
for label in CONFIGS:
    ax.plot(sys_lmp[label].index, sys_lmp[label].values,
            linewidth=0.3, alpha=0.6, color=CONFIG_COLORS[label], label=label)
ax.set_ylabel('$/MWh')
ax.set_title('Hourly Load-Weighted LMP')
ax.legend()

# (b) Daily average LMP
ax = axes[0, 1]
for label in CONFIGS:
    daily_lmp = sys_lmp[label].groupby(sys_lmp[label].index.normalize()).mean()
    ax.plot(daily_lmp.index, daily_lmp.values,
            linewidth=1.2, color=CONFIG_COLORS[label], label=label)
ax.set_ylabel('$/MWh')
ax.set_title('Daily Average LMP')
ax.legend()

# (c) LMP histogram
ax = axes[1, 0]
for label in CONFIGS:
    ax.hist(sys_lmp[label].values, bins=80, alpha=0.5,
            color=CONFIG_COLORS[label], label=label, edgecolor='white', linewidth=0.3)
ax.set_xlabel('$/MWh')
ax.set_ylabel('Hours')
ax.set_title('LMP Distribution')
ax.legend()

# (d) Average hourly profile
ax = axes[1, 1]
for label in CONFIGS:
    profile = sys_lmp[label].groupby(sys_lmp[label].index.hour).mean()
    ax.plot(profile.index, profile.values, linewidth=2,
            color=CONFIG_COLORS[label], label=label)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('$/MWh')
ax.set_title('Average Hourly LMP Profile')
ax.set_xlim(0, 23)
ax.set_xticks(range(0, 24, 4))
ax.legend()

plt.suptitle(f'LMP Analysis — {N_DAYS}-Day Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Supply–Demand Balance

In [ ]:
fig, axes = plt.subplots(len(CONFIGS), 1, figsize=(14, 4 * len(CONFIGS)), sharex=True)
if len(CONFIGS) == 1:
    axes = [axes]

for ax, label in zip(axes, CONFIGS):
    hg = pcm[label]['hourly_gen']
    hs = pcm[label]['hourly_summary']

    demand = hs['Demand'].values
    dt = hs['Datetime']

    ax.plot(dt, demand, linewidth=0.5, color='black', label='Demand')

    shed_mask = hg['Load shedding'] > 0
    if shed_mask.any():
        ax.scatter(hg.loc[shed_mask, 'Datetime'], demand[shed_mask.values],
                   color='red', s=15, zorder=5, label='Load Shedding')

    overgen_mask = hg['Over generation'] > 100
    if overgen_mask.any():
        ax.scatter(hg.loc[overgen_mask, 'Datetime'], demand[overgen_mask.values],
                   color='blue', s=5, alpha=0.4, zorder=4, label='Over-gen > 100MW')

    ax.set_ylabel('MW')
    ax.set_title(f'{label} — System Demand')
    ax.legend(fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.show()

## 7. Operational Violations

Comprehensive analysis of operational violations:
- Load shedding (unserved energy)
- Over-generation
- Reserve shortfall
- Transmission line violations
- Negative LMP events (economic signal of congestion/over-supply)

In [ ]:
print(f'Operational Violations Summary — {N_DAYS} days')
print('=' * 80)

violation_rows = []
for label in CONFIGS:
    o = pcm[label]['overall'].iloc[0]
    hg = pcm[label]['hourly_gen']
    line = pcm[label]['line']

    line_viol = line[line['Violation'] > 0]
    lines_with_viol = line_viol['Line'].nunique() if len(line_viol) > 0 else 0

    violation_rows.append({
        'Config': label,
        'Load Shedding (MWh)': o['Total load shedding'],
        'Load Shed Hours': int((hg['Load shedding'] > 0).sum()),
        'Over-Generation (MWh)': o['Total over generation'],
        'Over-Gen Hours': int((hg['Over generation'] > 0).sum()),
        'Reserve Shortfall (MWh)': o['Total reserve shortfall'],
        'Reserve Shortfall Hours': int((hg['Reserve shortfall'] > 0).sum()),
        'Line Violation Events': len(line_viol),
        'Lines with Violations': lines_with_viol,
        'Total Line Violation (MW)': line_viol['Violation'].sum() if len(line_viol) > 0 else 0,
        'Negative LMP Hours': int((sys_lmp[label] < 0).sum()),
    })

viol_df = pd.DataFrame(violation_rows).set_index('Config').T
for metric in viol_df.index:
    vals = viol_df.loc[metric]
    print(f'  {metric:<30s}', end='')
    for v in vals:
        if isinstance(v, float):
            print(f'{v:>14,.2f}', end='')
        else:
            print(f'{v:>14,}', end='')
    print()

In [ ]:
# Detailed over-generation analysis
print('Over-Generation Analysis')
print('=' * 70)

for label in CONFIGS:
    hg = pcm[label]['hourly_gen'].copy()
    overgen = hg[hg['Over generation'] > 0]
    if len(overgen) == 0:
        print(f'\n{label}: No over-generation')
        continue

    print(f'\n{label}:')
    print(f'  Hours with over-generation: {len(overgen)} / {len(hg)} ({len(overgen)/len(hg)*100:.1f}%)')
    print(f'  Total: {overgen["Over generation"].sum():,.1f} MWh')
    print(f'  Mean (when > 0): {overgen["Over generation"].mean():,.1f} MW')
    print(f'  Max: {overgen["Over generation"].max():,.1f} MW')

    overgen = overgen.copy()
    overgen['Hour'] = overgen['Datetime'].dt.hour
    by_hour = overgen.groupby('Hour')['Over generation'].agg(['count', 'sum', 'mean'])
    by_hour.columns = ['Hours', 'Total (MWh)', 'Mean (MW)']
    print(f'\n  By hour of day:')
    print(by_hour.to_string())

In [ ]:
# Reserve shortfall analysis
print('Reserve Shortfall Analysis')
print('=' * 70)

for label in CONFIGS:
    hg = pcm[label]['hourly_gen'].copy()
    shortfall = hg[hg['Reserve shortfall'] > 0]
    if len(shortfall) == 0:
        print(f'\n{label}: No reserve shortfall')
        continue

    print(f'\n{label}:')
    print(f'  Hours with shortfall: {len(shortfall)} / {len(hg)} ({len(shortfall)/len(hg)*100:.2f}%)')
    print(f'  Total: {shortfall["Reserve shortfall"].sum():,.2f} MWh')
    print(f'  Mean (when > 0): {shortfall["Reserve shortfall"].mean():,.2f} MW')
    print(f'  Max: {shortfall["Reserve shortfall"].max():,.2f} MW')

    shortfall = shortfall.copy()
    shortfall['Hour'] = shortfall['Datetime'].dt.hour
    by_hour = shortfall.groupby('Hour')['Reserve shortfall'].agg(['count', 'sum'])
    by_hour.columns = ['Hours', 'Total (MWh)']
    print(f'\n  By hour of day:')
    print(by_hour.to_string())

In [ ]:
# Transmission line violation analysis
print('Transmission Line Violation Analysis')
print('=' * 80)

for label in CONFIGS:
    line = pcm[label]['line']
    violations = line[line['Violation'] > 0]

    if len(violations) == 0:
        print(f'\n{label}: No line violations')
        continue

    print(f'\n{label}:')
    print(f'  Total violation events: {len(violations):,}')
    print(f'  Lines with violations: {violations["Line"].nunique()}')
    print(f'  Total violation magnitude: {violations["Violation"].sum():,.2f} MW')
    print(f'  Max single violation: {violations["Violation"].max():,.2f} MW')

    top_lines = (
        violations.groupby('Line')['Violation']
        .agg(['count', 'sum', 'max'])
        .sort_values('sum', ascending=False)
        .head(10)
    )
    top_lines.columns = ['Events', 'Total Viol (MW)', 'Max Viol (MW)']
    print(f'\n  Top 10 most violated lines:')
    print(top_lines.to_string())

In [ ]:
# Negative LMP analysis (economic violation / congestion signal)
print('Negative LMP Analysis')
print('=' * 70)

for label in CONFIGS:
    bus = pcm[label]['bus']
    neg_bus = bus[bus['LMP DA'] < 0]
    total_bus_hours = len(bus)

    print(f'\n{label}:')
    print(f'  Bus-hours with negative LMP: {len(neg_bus):,} / {total_bus_hours:,} '
          f'({len(neg_bus)/total_bus_hours*100:.3f}%)')
    print(f'  System-hours with negative LW-LMP: {(sys_lmp[label] < 0).sum()}')

    if len(neg_bus) > 0:
        print(f'  Min LMP: ${neg_bus["LMP DA"].min():.2f}/MWh')
        print(f'  Buses with negative LMP: {neg_bus["Bus"].nunique()}')

        top_buses = neg_bus.groupby('Bus').size().sort_values(ascending=False).head(5)
        print(f'  Top 5 buses by negative LMP hours:')
        for bus_name, cnt in top_buses.items():
            print(f'    {bus_name}: {cnt} hours')

In [ ]:
# Violation timeseries plots
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# (a) Over-generation
ax = axes[0]
for label in CONFIGS:
    hg = pcm[label]['hourly_gen']
    ax.plot(hg['Datetime'], hg['Over generation'],
            linewidth=0.5, alpha=0.6, color=CONFIG_COLORS[label], label=label)
ax.set_ylabel('MW')
ax.set_title('Hourly Over-Generation')
ax.legend(fontsize=8)

# (b) Reserve shortfall
ax = axes[1]
for label in CONFIGS:
    hg = pcm[label]['hourly_gen']
    ax.plot(hg['Datetime'], hg['Reserve shortfall'],
            linewidth=0.5, alpha=0.6, color=CONFIG_COLORS[label], label=label)
ax.set_ylabel('MW')
ax.set_title('Hourly Reserve Shortfall')
ax.legend(fontsize=8)

# (c) Available headroom
ax = axes[2]
for label in CONFIGS:
    hg = pcm[label]['hourly_gen']
    ax.plot(hg['Datetime'], hg['Available headroom'],
            linewidth=0.5, alpha=0.6, color=CONFIG_COLORS[label], label=label)
ax.set_ylabel('MW')
ax.set_title('Available Headroom')
ax.legend(fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.suptitle(f'Operational Violations — {N_DAYS}-Day Comparison',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Renewables Curtailment

In [ ]:
print(f'Renewables Curtailment Analysis — {N_DAYS} days')
print('=' * 70)

for label in CONFIGS:
    rnw = pcm[label]['renewables'].copy()
    gen = pcm[label]['gen']
    gen_info = gen[['GEN UID', 'Unit Type']].copy()
    gen_info['GEN UID'] = gen_info['GEN UID'].astype(str)
    rnw['Generator'] = rnw['Generator'].astype(str)
    rnw = rnw.merge(gen_info, left_on='Generator', right_on='GEN UID', how='left')

    total_curt = rnw['Curtailment'].sum()
    total_output = rnw['Output'].sum()
    total_avail = total_output + total_curt

    print(f'\n{label}:')
    print(f'  Total available: {total_avail/1e3:,.1f} GWh')
    print(f'  Total used: {total_output/1e3:,.1f} GWh')
    print(f'  Total curtailed: {total_curt/1e3:,.1f} GWh ({total_curt/total_avail*100:.2f}%)')

    by_type = rnw.groupby('Unit Type').agg(
        output=('Output', 'sum'),
        curtailment=('Curtailment', 'sum'),
    )
    by_type['total'] = by_type['output'] + by_type['curtailment']
    by_type['curt_pct'] = by_type['curtailment'] / by_type['total'] * 100
    by_type = by_type.round(1)
    print(f'  By type:')
    for ut, row in by_type.iterrows():
        print(f'    {ut}: output={row["output"]/1e3:.1f} GWh, '
              f'curtailed={row["curtailment"]/1e3:.1f} GWh ({row["curt_pct"]:.2f}%)')

In [ ]:
# Daily curtailment plot
fig, ax = plt.subplots(figsize=(14, 5))

for label in CONFIGS:
    rnw = pcm[label]['renewables'].copy()
    rnw['date'] = rnw['Datetime'].dt.normalize()
    daily_curt = rnw.groupby('date')['Curtailment'].sum() / 1e3
    ax.plot(daily_curt.index, daily_curt.values,
            linewidth=1.2, color=CONFIG_COLORS[label], label=label)

ax.set_ylabel('Curtailment (GWh/day)')
ax.set_title(f'Daily Renewables Curtailment — {N_DAYS}-Day Comparison')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.show()

## 9. Computational Performance

In [ ]:
print(f'Computational Performance — {N_DAYS} days')
print('=' * 75)

print(f'\n{"":<8s} {"":<6s}', end='')
for label in CONFIGS:
    print(f'{label:>14s}', end='')
print()
print('-' * 55)

for solve_type in ['SCED', 'RUC']:
    for stat_name, stat_fn in [('Count', 'count'), ('Mean (s)', 'mean'),
                                ('Median (s)', 'median'), ('Max (s)', 'max')]:
        print(f'{solve_type:<8s} {stat_name:<10s}', end='')
        for label in CONFIGS:
            rt = pcm[label]['runtimes']
            times = rt[rt['Type'] == solve_type]['Solve Time']
            val = getattr(times, stat_fn)() if len(times) > 0 else 0
            print(f'{val:>14.3f}', end='')
        print()
    print()

print('Total solver time (s):')
for label in CONFIGS:
    rt = pcm[label]['runtimes']
    total = rt['Solve Time'].sum()
    print(f'  {label}: {total:,.0f} s ({total/3600:.1f} hours)')

## 10. Summary

In [ ]:
print(f'PCM Commitment Period Comparison — {N_DAYS}-Day Summary')
print('=' * 80)

# Compact summary table
summary_rows = []
for label in CONFIGS:
    o = pcm[label]['overall'].iloc[0]
    hg = pcm[label]['hourly_gen']
    bus = pcm[label]['bus']
    line_viol = pcm[label]['line'][pcm[label]['line']['Violation'] > 0]
    lw = (bus['Demand'] * bus['LMP DA']).sum() / bus['Demand'].sum()

    # GTEP load shedding
    gtep_ls = sum(v for v in gtep[label].get('load_shed', {}).values()
                  if isinstance(v, (int, float)))

    summary_rows.append({
        'Config': label,
        'GTEP Load Shed (MWh)': f'{gtep_ls:,.0f}',
        'GTEP Retirements': sum(1 for v in gtep[label].get('dispatchable_investments', {}).values() if v is False),
        'PCM LW-LMP ($/MWh)': f'{lw:.2f}',
        'PCM Load Shed (MWh)': f'{o["Total load shedding"]:.1f}',
        'PCM Over-Gen (MWh)': f'{o["Total over generation"]:,.0f}',
        'PCM Curtailment (MWh)': f'{o["Total renewables curtailment"]:,.0f}',
        'PCM Reserve Short (MWh)': f'{o["Total reserve shortfall"]:.1f}',
        'PCM Line Violations': f'{len(line_viol):,}',
        'PCM Neg LMP Hours': f'{(sys_lmp[label] < 0).sum()}',
        'Renew Pen (%)': f'{o["Overall renewables penetration rate"]:.1f}',
    })

summary_df = pd.DataFrame(summary_rows).set_index('Config').T
for metric in summary_df.index:
    vals = summary_df.loc[metric]
    print(f'  {metric:<30s}', end='')
    for v in vals:
        print(f'{str(v):>14s}', end='')
    print()

# Overall assessment
print('\n── Assessment ──')
all_same = all(
    gtep[l1].get('dispatchable_investments') == gtep[l2].get('dispatchable_investments')
    for l1, l2 in [('1HR', '2HR'), ('2HR', '4HR')]
)
if all_same:
    print('  GTEP investment decisions are identical across all configs.')
    print('  PCM results reflect the same generator fleet.')
    print('  Differences will emerge with real cost data that drives differentiated retirements.')
else:
    print('  GTEP investment decisions differ — PCM results reflect different generator fleets.')